In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

# ─── ✏️  MUST MATCH 01_Bronze_Layer settings ─────────────────────────────────
CATALOG        = "main"
DATABASE       = "flight_tracker"
BRONZE_TABLE   = f"{CATALOG}.{DATABASE}.flight_raw"
SILVER_CLEAN   = f"{CATALOG}.{DATABASE}.flight_clean"
SILVER_CURRENT = f"{CATALOG}.{DATABASE}.flight_current"
SILVER_HISTORY = f"{CATALOG}.{DATABASE}.flight_history"
SILVER_CDC     = f"{CATALOG}.{DATABASE}.flight_cdc"

# Cleaning thresholds (from your config.yaml)
LAT_MIN, LAT_MAX      = -90.0,   90.0
LON_MIN, LON_MAX      = -180.0, 180.0
ALT_MIN_FT            = -1000
ALT_MAX_FT            =  60000
SPEED_MAX_KT          =  1200

# CDC thresholds
POSITION_DELTA        =  0.001   # degrees (~111 m)
ALT_DELTA_FT          =  100
SPEED_DELTA_KT        =  10
HEADING_DELTA         =  5

EMERGENCY_SQUAWKS     = ("7500", "7600", "7700")
print("✅ Silver config loaded")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_CLEAN} (
    ingestion_id      STRING,
    fetched_at        TIMESTAMP  NOT NULL,
    source_region     STRING,
    hex               STRING     NOT NULL,
    callsign          STRING     COMMENT 'UNKNOWN when original is NULL/empty',
    lat               DOUBLE,
    lon               DOUBLE,
    alt_baro          INT        COMMENT 'NULL when out of range',
    gs                DOUBLE     COMMENT 'NULL when absurd speed',
    track             DOUBLE,
    baro_rate         INT,
    squawk            STRING,
    registration      STRING     COMMENT 'UNREGISTERED when original is NULL',
    aircraft_type     STRING,
    airline_icao      STRING     COMMENT 'First 3 chars of callsign',
    is_emergency      BOOLEAN,
    is_military       BOOLEAN,
    -- Soft Delete columns ────────────────────────────────────────────────────
    is_deleted        BOOLEAN    COMMENT 'TRUE = logically deleted, never physically removed',
    deleted_at        TIMESTAMP  COMMENT 'When this row was soft-deleted',
    deleted_reason    STRING     COMMENT 'INVALID_COORDS | DUPLICATE | DISAPPEARED | MANUAL',
    -- Audit ───────────────────────────────────────────────────────────────────
    _silver_loaded_at TIMESTAMP
)
USING DELTA
COMMENT 'Silver: cleaned observations with soft delete — query WHERE is_deleted=FALSE for live data'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'delta.enableChangeDataFeed'       = 'true'
)
""")
print(f"✅ {SILVER_CLEAN}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_CURRENT} (
    hex               STRING     NOT NULL,
    callsign          STRING,
    lat               DOUBLE,
    lon               DOUBLE,
    alt_baro          INT,
    gs                DOUBLE,
    track             DOUBLE,
    baro_rate         INT,
    squawk            STRING,
    registration      STRING,
    aircraft_type     STRING,
    airline_icao      STRING,
    source_region     STRING,
    is_emergency      BOOLEAN,
    is_military       BOOLEAN,
    last_seen_at      TIMESTAMP,
    -- Soft Delete columns ────────────────────────────────────────────────────
    is_deleted        BOOLEAN    COMMENT 'TRUE = aircraft no longer observed (DISAPPEARED)',
    deleted_at        TIMESTAMP,
    deleted_reason    STRING,
    -- Audit ───────────────────────────────────────────────────────────────────
    _updated_at       TIMESTAMP
)
USING DELTA
COMMENT 'Silver: live snapshot — one row per aircraft. Filter is_deleted=FALSE for active.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed'       = 'true'
)
""")
print(f"✅ {SILVER_CURRENT}")

In [0]:
# ── flight_history (SCD2) ─────────────────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_HISTORY} (
    flight_sk         BIGINT     GENERATED ALWAYS AS IDENTITY,
    hex               STRING     NOT NULL,
    callsign          STRING,
    registration      STRING,
    aircraft_type     STRING,
    airline_icao      STRING,
    effective_from    TIMESTAMP  NOT NULL,
    effective_to      TIMESTAMP  COMMENT 'NULL while is_current=TRUE',
    is_current        BOOLEAN    NOT NULL,
    -- Soft Delete ─────────────────────────────────────────────────────────────
    is_deleted        BOOLEAN,
    deleted_at        TIMESTAMP,
    deleted_reason    STRING
)
USING DELTA
COMMENT 'Silver SCD2: full history of aircraft identity changes'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
""")
print(f"✅ {SILVER_HISTORY}")

# ── flight_cdc ─────────────────────────────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_CDC} (
    cdc_id            BIGINT     GENERATED ALWAYS AS IDENTITY,
    hex               STRING     NOT NULL,
    callsign          STRING,
    change_type       STRING     NOT NULL  COMMENT 'NEW|POSITION|ALTITUDE|SPEED|HEADING|DISAPPEARED',
    old_value         STRING,
    new_value         STRING,
    detected_at       TIMESTAMP  NOT NULL
)
USING DELTA
COMMENT 'Silver CDC: event log of all detected aircraft state changes'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
""")
print(f"✅ {SILVER_CDC}")

In [0]:

def clean_bronze_to_silver():
    # High-water mark: last fetched_at already in Silver
    hwm = spark.sql(f"""
        SELECT COALESCE(MAX(fetched_at), CAST('1970-01-01' AS TIMESTAMP))
        FROM {SILVER_CLEAN}
    """).collect()[0][0]
    print(f"📍 High-water mark: {hwm}")

    # New Bronze rows above HWM, with duplicate rank
    new_bronze = spark.sql(f"""
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY hex, fetched_at
                   ORDER BY seen NULLS LAST
               ) AS _rn
        FROM {BRONZE_TABLE}
        WHERE fetched_at > '{hwm}'
          AND hex IS NOT NULL
    """)

    cleaned = new_bronze.select(
        F.col("ingestion_id"),
        F.col("fetched_at"),
        F.col("source_region"),
        F.col("hex"),

        # Callsign: NULL/empty → 'UNKNOWN'
        F.coalesce(
            F.when(F.trim(F.col("flight")) != "", F.trim(F.col("flight"))),
            F.lit("UNKNOWN")
        ).alias("callsign"),

        F.col("lat"),
        F.col("lon"),

        # Altitude: out-of-range → NULL
        F.when(
            F.col("alt_baro").between(ALT_MIN_FT, ALT_MAX_FT),
            F.col("alt_baro")
        ).otherwise(F.lit(None).cast("int")).alias("alt_baro"),

        # Ground speed: absurd → NULL
        F.when(
            (F.col("gs") >= 0) & (F.col("gs") <= SPEED_MAX_KT),
            F.col("gs")
        ).otherwise(F.lit(None).cast("double")).alias("gs"),

        F.col("track"),
        F.col("baro_rate"),
        F.col("squawk"),

        # Registration: NULL → 'UNREGISTERED'
        F.coalesce(F.col("registration"), F.lit("UNREGISTERED")).alias("registration"),
        F.col("aircraft_type"),

        # Airline ICAO: first 3 chars of callsign
        F.when(
            F.col("flight").isNotNull() & (F.length(F.trim(F.col("flight"))) >= 3),
            F.upper(F.substring(F.trim(F.col("flight")), 1, 3))
        ).otherwise(F.lit(None)).alias("airline_icao"),

        # is_emergency
        (
            F.col("squawk").isin(list(EMERGENCY_SQUAWKS)) |
            (F.col("emergency").isNotNull() & ~F.col("emergency").isin("none", ""))
        ).alias("is_emergency"),

        # is_military: bit 0 of db_flags
        ((F.coalesce(F.col("db_flags"), F.lit(0)).bitwiseAND(1)) == 1
        ).alias("is_military"),

        # ── Soft Delete: flag invalid rows with reason ──────────────────────
        F.when(
            F.col("lat").isNull() | F.col("lon").isNull() |
            ~F.col("lat").between(LAT_MIN, LAT_MAX) |
            ~F.col("lon").between(LON_MIN, LON_MAX),
            F.lit(True)
        ).when(F.col("_rn") > 1, F.lit(True)
        ).otherwise(F.lit(False)).alias("is_deleted"),

        F.when(
            F.col("lat").isNull() | F.col("lon").isNull() |
            ~F.col("lat").between(LAT_MIN, LAT_MAX) |
            ~F.col("lon").between(LON_MIN, LON_MAX),
            F.lit("INVALID_COORDS")
        ).when(F.col("_rn") > 1, F.lit("DUPLICATE")
        ).otherwise(F.lit(None)).alias("deleted_reason"),

        F.when(
            F.col("lat").isNull() | F.col("lon").isNull() |
            ~F.col("lat").between(LAT_MIN, LAT_MAX) |
            ~F.col("lon").between(LON_MIN, LON_MAX) |
            (F.col("_rn") > 1),
            F.current_timestamp()
        ).otherwise(F.lit(None).cast("timestamp")).alias("deleted_at"),

        F.current_timestamp().alias("_silver_loaded_at"),
    )

    total   = cleaned.count()
    deleted = cleaned.filter(F.col("is_deleted") == True).count()
    active  = total - deleted

    cleaned.write.format("delta").mode("append").saveAsTable(SILVER_CLEAN)

    print(f"✅ Silver flight_clean:")
    print(f"   Total    : {total:,}")
    print(f"   Active   : {active:,}")
    print(f"   Soft-del : {deleted:,}  (flagged is_deleted=TRUE, reason code set)")
    return total

In [0]:
def rebuild_flight_current():
    """
    Upsert latest valid position per aircraft into flight_current.
    Aircraft that vanish from the latest batch are soft-deleted (DISAPPEARED),
    not physically removed.
    """
    # Latest valid observation per aircraft
    latest_df = spark.sql(f"""
        SELECT hex, callsign, lat, lon, alt_baro, gs, track, baro_rate,
               squawk, registration, aircraft_type, airline_icao,
               source_region, is_emergency, is_military, fetched_at AS last_seen_at,
               ROW_NUMBER() OVER (PARTITION BY hex ORDER BY fetched_at DESC) AS rn
        FROM {SILVER_CLEAN}
        WHERE is_deleted = FALSE
    """).filter("rn = 1").drop("rn")

    # Try MERGE (requires table to already exist and have rows)
    try:
        current_dt = DeltaTable.forName(spark, SILVER_CURRENT)

        # 1) Upsert active aircraft
        (current_dt.alias("tgt")
            .merge(latest_df.alias("src"), "tgt.hex = src.hex")
            .whenMatchedUpdate(set={
                "callsign":       "src.callsign",
                "lat":            "src.lat",
                "lon":            "src.lon",
                "alt_baro":       "src.alt_baro",
                "gs":             "src.gs",
                "track":          "src.track",
                "baro_rate":      "src.baro_rate",
                "squawk":         "src.squawk",
                "registration":   "src.registration",
                "aircraft_type":  "src.aircraft_type",
                "airline_icao":   "src.airline_icao",
                "source_region":  "src.source_region",
                "is_emergency":   "src.is_emergency",
                "is_military":    "src.is_military",
                "last_seen_at":   "src.last_seen_at",
                "is_deleted":     "false",
                "deleted_at":     "null",
                "deleted_reason": "null",
                "_updated_at":    "current_timestamp()",
            })
            .whenNotMatchedInsert(values={
                "hex":            "src.hex",
                "callsign":       "src.callsign",
                "lat":            "src.lat",
                "lon":            "src.lon",
                "alt_baro":       "src.alt_baro",
                "gs":             "src.gs",
                "track":          "src.track",
                "baro_rate":      "src.baro_rate",
                "squawk":         "src.squawk",
                "registration":   "src.registration",
                "aircraft_type":  "src.aircraft_type",
                "airline_icao":   "src.airline_icao",
                "source_region":  "src.source_region",
                "is_emergency":   "src.is_emergency",
                "is_military":    "src.is_military",
                "last_seen_at":   "src.last_seen_at",
                "is_deleted":     "false",
                "deleted_at":     "null",
                "deleted_reason": "null",
                "_updated_at":    "current_timestamp()",
            })
            .execute()
        )

        # 2) Soft-delete aircraft absent from the latest fetch batch
        latest_ts = spark.sql(
            f"SELECT MAX(fetched_at) FROM {SILVER_CLEAN} WHERE is_deleted=FALSE"
        ).collect()[0][0]

        spark.sql(f"""
            UPDATE {SILVER_CURRENT}
            SET    is_deleted     = true,
                   deleted_at    = current_timestamp(),
                   deleted_reason = 'DISAPPEARED',
                   _updated_at   = current_timestamp()
            WHERE  is_deleted = false
              AND  hex NOT IN (
                SELECT hex FROM {SILVER_CLEAN}
                WHERE  is_deleted = false
                  AND  fetched_at = '{latest_ts}'
              )
        """)
        print("   Used MERGE strategy")

    except Exception:
        # First run or empty table — plain insert
        (latest_df
            .withColumn("is_deleted",     F.lit(False))
            .withColumn("deleted_at",     F.lit(None).cast("timestamp"))
            .withColumn("deleted_reason", F.lit(None).cast("string"))
            .withColumn("_updated_at",    F.current_timestamp())
            .write.format("delta").mode("overwrite").saveAsTable(SILVER_CURRENT)
        )
        print("   Used initial INSERT strategy (first run)")

    active = spark.sql(f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=FALSE").collect()[0][0]
    gone   = spark.sql(f"SELECT COUNT(*) FROM {SILVER_CURRENT} WHERE is_deleted=TRUE").collect()[0][0]
    print(f"✅ flight_current: {active:,} active | {gone:,} soft-deleted (DISAPPEARED)")

In [0]:

def apply_scd2():
    """
    Tracks changes to an aircraft's identity attributes.
    When callsign/registration/type changes:
      - Old row: effective_to=now, is_current=FALSE
      - New row: effective_from=now, is_current=TRUE
    """
    TRACKED = ["callsign", "registration", "aircraft_type", "airline_icao"]

    current_df = spark.sql(f"""
        SELECT hex, callsign, registration, aircraft_type, airline_icao, last_seen_at
        FROM {SILVER_CURRENT}
        WHERE is_deleted = FALSE
    """)

    history_open = spark.sql(f"""
        SELECT hex, callsign, registration, aircraft_type, airline_icao
        FROM {SILVER_HISTORY}
        WHERE is_current = TRUE AND is_deleted = FALSE
    """)

    changed_df = (current_df.alias("cur")
        .join(history_open.alias("hist"), on="hex", how="left")
        .filter(
            F.col("hist.hex").isNull() |
            (F.col("cur.callsign")      != F.col("hist.callsign")) |
            (F.col("cur.registration")  != F.col("hist.registration")) |
            (F.col("cur.aircraft_type") != F.col("hist.aircraft_type")) |
            (F.col("cur.airline_icao")  != F.col("hist.airline_icao"))
        )
        .select("cur.hex","cur.callsign","cur.registration",
                "cur.aircraft_type","cur.airline_icao","cur.last_seen_at")
    )

    n = changed_df.count()
    if n == 0:
        print("✅ SCD2: No identity changes")
        return 0

    hexes = [r.hex for r in changed_df.select("hex").collect()]
    hex_in = ", ".join(f"'{h}'" for h in hexes)

    # Close old rows
    spark.sql(f"""
        UPDATE {SILVER_HISTORY}
        SET effective_to = current_timestamp(), is_current = FALSE
        WHERE is_current = TRUE AND hex IN ({hex_in})
    """)

    # Open new rows
    (changed_df
        .withColumn("effective_from", F.current_timestamp())
        .withColumn("effective_to",   F.lit(None).cast("timestamp"))
        .withColumn("is_current",     F.lit(True))
        .withColumn("is_deleted",     F.lit(False))
        .withColumn("deleted_at",     F.lit(None).cast("timestamp"))
        .withColumn("deleted_reason", F.lit(None).cast("string"))
        .drop("last_seen_at")
        .write.format("delta").mode("append").saveAsTable(SILVER_HISTORY)
    )
    print(f"✅ SCD2: {n:,} new identity versions opened")
    return n

In [0]:
def detect_cdc_changes():
    """Compares latest vs previous observation per aircraft. Logs meaningful changes."""
    hwm = spark.sql(
        f"SELECT COALESCE(MAX(detected_at), CAST('1970-01-01' AS TIMESTAMP)) FROM {SILVER_CDC}"
    ).collect()[0][0]

    base_cte = f"""
        WITH ranked AS (
            SELECT hex, callsign, lat, lon, alt_baro, gs, track, fetched_at,
                   ROW_NUMBER() OVER (PARTITION BY hex ORDER BY fetched_at DESC) AS rn
            FROM {SILVER_CLEAN}
            WHERE is_deleted = FALSE
        ),
        cur  AS (SELECT * FROM ranked WHERE rn = 1),
        prev AS (SELECT * FROM ranked WHERE rn = 2)
    """

    sqls = {
        "NEW": f"""{base_cte}
            SELECT cur.hex, cur.callsign, 'NEW',
                   NULL, CONCAT('lat=',cur.lat,',lon=',cur.lon), cur.fetched_at
            FROM cur LEFT JOIN prev USING (hex)
            WHERE prev.hex IS NULL AND cur.fetched_at > '{hwm}'""",

        "POSITION": f"""{base_cte}
            SELECT cur.hex, cur.callsign, 'POSITION',
                   CONCAT(prev.lat,',',prev.lon), CONCAT(cur.lat,',',cur.lon), cur.fetched_at
            FROM cur JOIN prev USING (hex)
            WHERE cur.fetched_at > '{hwm}'
              AND (ABS(cur.lat-prev.lat)>{POSITION_DELTA} OR ABS(cur.lon-prev.lon)>{POSITION_DELTA})""",

        "ALTITUDE": f"""{base_cte}
            SELECT cur.hex, cur.callsign, 'ALTITUDE',
                   CAST(prev.alt_baro AS STRING), CAST(cur.alt_baro AS STRING), cur.fetched_at
            FROM cur JOIN prev USING (hex)
            WHERE cur.fetched_at > '{hwm}'
              AND ABS(COALESCE(cur.alt_baro,0)-COALESCE(prev.alt_baro,0)) > {ALT_DELTA_FT}""",

        "SPEED": f"""{base_cte}
            SELECT cur.hex, cur.callsign, 'SPEED',
                   CAST(prev.gs AS STRING), CAST(cur.gs AS STRING), cur.fetched_at
            FROM cur JOIN prev USING (hex)
            WHERE cur.fetched_at > '{hwm}'
              AND ABS(COALESCE(cur.gs,0)-COALESCE(prev.gs,0)) > {SPEED_DELTA_KT}""",

        "HEADING": f"""{base_cte}
            SELECT cur.hex, cur.callsign, 'HEADING',
                   CAST(prev.track AS STRING), CAST(cur.track AS STRING), cur.fetched_at
            FROM cur JOIN prev USING (hex)
            WHERE cur.fetched_at > '{hwm}'
              AND ABS(COALESCE(cur.track,0)-COALESCE(prev.track,0)) > {HEADING_DELTA}""",

        "DISAPPEARED": f"""{base_cte}
            SELECT prev.hex, prev.callsign, 'DISAPPEARED',
                   CONCAT('lat=',prev.lat,',lon=',prev.lon), NULL, current_timestamp()
            FROM prev LEFT JOIN cur USING (hex)
            WHERE cur.hex IS NULL AND prev.fetched_at > '{hwm}'""",
    }

    from functools import reduce
    events = [spark.sql(s).toDF("hex","callsign","change_type","old_value","new_value","detected_at")
              for s in sqls.values()]
    all_events = reduce(lambda a, b: a.union(b), events)
    n = all_events.count()

    if n > 0:
        all_events.write.format("delta").mode("append").saveAsTable(SILVER_CDC)
    print(f"✅ CDC: {n:,} change events since {hwm}")
    return n

In [0]:
def soft_delete_aircraft(hex_code: str, reason: str = "MANUAL"):
    """Mark a specific aircraft as logically deleted in flight_current and flight_history."""
    spark.sql(f"""
        UPDATE {SILVER_CURRENT}
        SET is_deleted=true, deleted_at=current_timestamp(),
            deleted_reason='{reason}', _updated_at=current_timestamp()
        WHERE hex='{hex_code}'
    """)
    spark.sql(f"""
        UPDATE {SILVER_HISTORY}
        SET effective_to=current_timestamp(), is_current=FALSE,
            is_deleted=true, deleted_at=current_timestamp(), deleted_reason='{reason}'
        WHERE hex='{hex_code}' AND is_current=TRUE
    """)
    print(f"🗑️  Soft-deleted: {hex_code}  (reason: {reason})")

def restore_aircraft(hex_code: str):
    """Restore a soft-deleted aircraft to active status."""
    spark.sql(f"""
        UPDATE {SILVER_CURRENT}
        SET is_deleted=false, deleted_at=null, deleted_reason=null,
            _updated_at=current_timestamp()
        WHERE hex='{hex_code}'
    """)
    print(f"✅ Restored: {hex_code}")

def view_soft_deleted():
    """Show all soft-deleted aircraft with their reasons."""
    return spark.sql(f"""
        SELECT hex, callsign, last_seen_at, deleted_at, deleted_reason
        FROM {SILVER_CURRENT}
        WHERE is_deleted = TRUE
        ORDER BY deleted_at DESC
    """)

# ── Example usage (uncomment to test) ────────────────────────────────────────
# soft_delete_aircraft("abc123", reason="TEST")
# restore_aircraft("abc123")
# view_soft_deleted().show()

print("✅ Soft delete utilities ready: soft_delete_aircraft(), restore_aircraft(), view_soft_deleted()")

In [0]:
try:
    clean_bronze_to_silver()
    rebuild_flight_current()
    apply_scd2()
    detect_cdc_changes()
except OverflowError as e:
    print(f"⚠️  Timestamp overflow error: {e}")
    print("\n💡 Root cause: Bronze table contains corrupted timestamps (year 58407+)")
    print("   This happened because the API 'now' field was in milliseconds but wasn't")
    print("   divided by 1000 during ingestion.")
    print("\n🔧 To fix:")
    print("   1. Truncate the Bronze table: spark.sql('TRUNCATE TABLE main.flight_tracker.flight_raw')")
    print("   2. Re-run the Bronze ingestion notebook (01_Bronze_ADSB) which now has the fix")
    print("   3. Then re-run this Silver pipeline")


In [0]:
print("=" * 60)
print("SILVER LAYER VERIFICATION")
print("=" * 60)

print("\n📊 flight_clean — Row counts")
spark.sql(f"""
    SELECT
        COUNT(*)                                              AS total_rows,
        SUM(CASE WHEN is_deleted=FALSE THEN 1 ELSE 0 END)    AS active,
        SUM(CASE WHEN is_deleted=TRUE  THEN 1 ELSE 0 END)    AS soft_deleted
    FROM {SILVER_CLEAN}
""").show()

print("\n🗑️  Soft delete reasons (flight_clean)")
spark.sql(f"""
    SELECT COALESCE(deleted_reason,'ACTIVE') AS reason, COUNT(*) AS count
    FROM {SILVER_CLEAN}
    GROUP BY 1 ORDER BY 2 DESC
""").show()

print("\n✈️  flight_current by region")
spark.sql(f"""
    SELECT source_region,
           SUM(CASE WHEN is_deleted=FALSE THEN 1 ELSE 0 END) AS active,
           SUM(CASE WHEN is_deleted=TRUE  THEN 1 ELSE 0 END) AS disappeared,
           SUM(CASE WHEN is_emergency     THEN 1 ELSE 0 END) AS emergencies
    FROM {SILVER_CURRENT}
    GROUP BY 1
""").show()

print("\n📜 SCD2 history versions")
spark.sql(f"SELECT is_current, COUNT(*) AS count FROM {SILVER_HISTORY} GROUP BY 1").show()

print("\n🔄 CDC event types")
spark.sql(f"""
    SELECT change_type, COUNT(*) AS count
    FROM {SILVER_CDC} GROUP BY 1 ORDER BY 2 DESC
""").show()

print("\n✈️  Sample active flights")
spark.sql(f"""
    SELECT hex, callsign, lat, lon, alt_baro, gs, source_region, last_seen_at
    FROM {SILVER_CURRENT}
    WHERE is_deleted = FALSE
    ORDER BY last_seen_at DESC LIMIT 10
""").show(truncate=False)

In [0]:
spark.sql(f"""
SELECT
    MIN(fetched_at),
    MAX(fetched_at)
FROM {SILVER_CLEAN}
""").show(truncate=False)

In [0]:
view_soft_deleted().show()